# ⚙️ VERSION 2 — yolo26s: KHÔNG train, chỉ val + export lại (nms=False, max_det=400, thêm 1024)
> Cấu hình đã đặt sẵn: `MODEL=yolo26s.pt`, `SKIP_TRAIN=True`, `EXPORT_SIZES=[640,960,1024,1280]`.
> **BẮT BUỘC Add Input = Output của Version 1** (notebookf47bf30fd3) trước khi Run All, nếu không cell 4 báo lỗi không tìm thấy best.pt. Thời gian ~15–25 phút.

# Đề tài 28 — Train YOLO26 trên VisDrone-DET (Kaggle)

**Việc duy nhất của notebook này:** train một detector YOLO26 (mặc định `yolo26s`) trên VisDrone2019-DET, đánh giá trên tập val, export OpenVINO/ONNX ở 3 độ phân giải, đóng gói thành 1 file zip để tải về laptop. Mọi bước khác của đề tài (dump trên VID/UAVDT, cue, thống kê) chạy trên laptop.

## Cài đặt Kaggle (làm TRƯỚC khi chạy)
1. **Settings → Accelerator:** `GPU P100` (nếu có) hoặc `GPU T4 x2`. Notebook chỉ dùng 1 GPU (`device=0`).
2. **Settings → Internet:** `On` (cần tài khoản Kaggle đã xác minh số điện thoại).
3. **Settings → Persistence:** `Files only`.
4. Chạy nền: **Save Version → Save & Run All (Commit)**. Giới hạn 12 h/phiên, 30 h GPU/tuần. Đóng trình duyệt được.
5. Kết quả: tab **Output** của version → `28_<NAME>_export.zip`.

## Nếu phiên bị cắt trước khi xong
Add **Output** của version trước làm **Input** (Add Input → Your Work → chọn notebook/version), rồi Save & Run All lại. Cell 4 tự tìm `last.pt` trong `/kaggle/input/**` và resume.

## Chạy model thứ hai (ví dụ `yolo26n`)
Đổi `MODEL` ở cell Cấu hình, Save Version mới. Mỗi model là một version riêng.


In [ ]:
# Cell 1 — cài đặt & thông tin máy
import subprocess, sys, os, json, time, hashlib, glob, shutil, platform, re
t_start = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics==8.4.163", "openvino==2026.4.0", "onnx", "onnxslim"], check=True)  # pinned: version smoke-tested (P2B)
import torch, ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU"


In [ ]:
# Cell 2 — cấu hình (chỉ sửa ở đây)
MODEL   = "yolo26s.pt"     # detector chính; đổi "yolo26n.pt" cho mức recall thấp (version riêng)
IMGSZ   = 1024
EPOCHS  = 80
BATCH   = -1               # autobatch theo VRAM
SEED    = 42
SMOKE   = False            # True = chạy thử 1 epoch nhỏ để bắt lỗi
EXPORT_SIZES = [640, 960, 1024, 1280]
SKIP_TRAIN = True        # True = không train; lấy best.pt từ Output version trước (Add Input) rồi chạy lại val + export
MAX_DET = 400             # số box tối đa/ảnh (VisDrone val có ảnh tới 317 object; export end-to-end cố định TopK = MAX_DET)

WORK    = "/kaggle/working"
DATASETS_DIR = f"{WORK}/datasets"
PROJECT = f"{WORK}/runs"
NAME    = f"{MODEL.replace('.pt','')}_visdrone_{IMGSZ}"
if SMOKE:
    EPOCHS, IMGSZ, BATCH = 1, 320, 8
    NAME += "_smoke"
os.makedirs(DATASETS_DIR, exist_ok=True); os.makedirs(PROJECT, exist_ok=True)

from ultralytics import settings
settings.update({"datasets_dir": DATASETS_DIR, "wandb": False, "mlflow": False, "comet": False, "tensorboard": False})
def sha256(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
print(json.dumps(dict(MODEL=MODEL, IMGSZ=IMGSZ, EPOCHS=EPOCHS, BATCH=BATCH, SEED=SEED, SMOKE=SMOKE, NAME=NAME, SKIP_TRAIN=SKIP_TRAIN, MAX_DET=MAX_DET), indent=1))


In [ ]:
# Cell 3 — dữ liệu VisDrone2019-DET (Ultralytics tự tải từ GitHub release; KHÔNG dùng Google Drive)
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import ASSETS_URL  # base URL of the VisDrone zips in VisDrone.yaml
data = check_det_dataset("VisDrone.yaml")   # tải + convert nếu chưa có
root = str(data["path"])
yaml_path = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "datasets", "VisDrone.yaml")
manifest = {"dataset": "VisDrone2019-DET", "yaml": "VisDrone.yaml", "root": root, "splits": {},
            "download_source": [u.replace("{ASSETS_URL}", ASSETS_URL) for u in re.findall(r"(?:https?://|\{ASSETS_URL\}/)[^\s\"']+", open(yaml_path, encoding="utf-8").read())]}  # yaml builds zip URLs from ASSETS_URL
for split in ["train", "val", "test"]:
    img_dir = data.get(split)
    if not img_dir: continue
    imgs = [f for f in glob.glob(os.path.join(img_dir, "*")) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    lbls = glob.glob(os.path.join(img_dir.replace("images", "labels"), "*.txt"))
    manifest["splits"][split] = {"images": len(imgs), "labels": len(lbls), "dir": img_dir}
zips = glob.glob(os.path.join(root, "*.zip")) + glob.glob(os.path.join(DATASETS_DIR, "*.zip"))
manifest["zips_sha256"] = {os.path.basename(z): sha256(z) for z in zips}
manifest["names"] = data["names"]
json.dump(manifest, open(f"{WORK}/dataset_manifest.json", "w"), indent=1)
print(json.dumps({k: v for k, v in manifest.items() if k != "names"}, indent=1))
assert manifest["splits"]["train"]["images"] > 6000, "VisDrone-DET train phải ~6.471 ảnh"


In [ ]:
# Cell 4 — train (tự resume nếu có last.pt từ Input của version trước)
run_dir = f"{PROJECT}/{NAME}"
if SKIP_TRAIN:
    # Không train: tìm best.pt của đúng NAME trong Output version trước (Add Input → Your Work)
    import zipfile
    os.makedirs(f"{run_dir}/weights", exist_ok=True)
    src_best = None
    hits = sorted(glob.glob(f"/kaggle/input/**/{NAME}/weights/best.pt", recursive=True))          # runs/<NAME>/weights/best.pt
    hits += sorted(p for p in glob.glob("/kaggle/input/**/weights/best.pt", recursive=True)        # zip đã được Kaggle giải nén
                   if NAME in p and p not in hits)
    if hits:
        src_best = hits[0]
        for f in ["best.pt", "last.pt"]:
            if os.path.exists(os.path.join(os.path.dirname(src_best), f)):
                shutil.copy2(os.path.join(os.path.dirname(src_best), f), f"{run_dir}/weights/{f}")
        for f in ["results.csv", "args.yaml"]:
            q = os.path.join(os.path.dirname(os.path.dirname(src_best)), f)
            if os.path.exists(q): shutil.copy2(q, f"{run_dir}/{f}")
    else:
        for z in sorted(glob.glob(f"/kaggle/input/**/28_{NAME}_export.zip", recursive=True)):         # zip nguyên
            with zipfile.ZipFile(z) as zf:
                if "weights/best.pt" not in zf.namelist(): continue
                for f in ["weights/best.pt", "weights/last.pt", "results.csv", "args.yaml"]:
                    if f in zf.namelist():
                        with zf.open(f) as s, open(f"{run_dir}/{f}", "wb") as d: shutil.copyfileobj(s, d)
                src_best = f"{z}::weights/best.pt"; break
    assert src_best and os.path.exists(f"{run_dir}/weights/best.pt"), (
        f"SKIP_TRAIN=True nhưng không tìm thấy best.pt của {NAME} trong /kaggle/input "
        f"(đã tìm **/{NAME}/weights/best.pt, **/weights/best.pt có '{NAME}' trong đường dẫn, **/28_{NAME}_export.zip). "
        "Hãy Add Input → Your Work → chọn version đã train xong, và kiểm MODEL/IMGSZ/SMOKE khớp với version đó.")
    print("SKIP_TRAIN: dùng best.pt từ", src_best)
    train_hours = float("nan")   # không train trong version này (xem TRAIN_LOG của version trước)
    best = f"{run_dir}/weights/best.pt"
else:
    last_local = f"{run_dir}/weights/last.pt"
    cands = glob.glob(f"/kaggle/input/**/{NAME}/weights/last.pt", recursive=True)
    if not os.path.exists(last_local) and cands:
        shutil.copytree(os.path.dirname(os.path.dirname(cands[0])), run_dir, dirs_exist_ok=True)
        print("Resume từ Input:", cands[0])

    train_kw = dict(data="VisDrone.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, seed=SEED, deterministic=True,
                    device=0, project=PROJECT, name=NAME, exist_ok=True, workers=4, patience=100,
                    fraction=(0.02 if SMOKE else 1.0), plots=True, verbose=True)
    t0 = time.time()
    resumed = False
    if os.path.exists(last_local):
        ck = torch.load(last_local, map_location="cpu", weights_only=False)
        if ck.get("epoch", -1) >= 0 and ck.get("epoch", -1) < EPOCHS - 1:
            print("RESUME từ epoch", ck.get("epoch"))
            model = YOLO(last_local); results = model.train(resume=True); resumed = True
    if not resumed and not os.path.exists(f"{run_dir}/weights/best.pt"):
        model = YOLO(MODEL); results = model.train(**train_kw)
    train_hours = (time.time() - t0) / 3600
    print(f"Train xong sau {train_hours:.2f} h")
    best = f"{run_dir}/weights/best.pt"; assert os.path.exists(best)


In [ ]:
# Cell 5 — val trên VisDrone-DET val (YOLO26 end-to-end, không NMS)
model = YOLO(best)
m = model.val(data="VisDrone.yaml", imgsz=IMGSZ, split="val", device=0, plots=True, project=PROJECT, name=NAME + "_val", exist_ok=True, nms=False, max_det=MAX_DET)  # nms=False = YOLO26 NMS-free head (ultralytics>=8.4.x default nms=None uses one-to-many head + NMS)
val_metrics = {
    "model": MODEL, "imgsz": IMGSZ, "epochs_requested": EPOCHS, "max_det": MAX_DET, "skip_train": SKIP_TRAIN,
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "precision": float(m.box.mp), "recall": float(m.box.mr),
    "per_class_mAP50_95": {model.names[int(i)]: float(m.box.maps[int(i)]) for i in m.box.ap_class_index},
    "speed_ms": m.speed,
}
json.dump(val_metrics, open(f"{WORK}/val_metrics.json", "w"), indent=1)
print(json.dumps(val_metrics, indent=1))


In [ ]:
# Cell 6 — export OpenVINO FP16 + ONNX cho 3 độ phân giải (chạy trên laptop Iris Xe / CPU)
exports = {}
os.makedirs(f"{WORK}/export", exist_ok=True)
for sz in EXPORT_SIZES:
    ov = YOLO(best).export(format="openvino", imgsz=sz, half=True, dynamic=False, nms=False, max_det=MAX_DET)  # nms=False -> end-to-end output (1,MAX_DET,6)
    ov_dst = f"{WORK}/export/{NAME}_openvino_{sz}"
    if os.path.exists(ov_dst): shutil.rmtree(ov_dst)
    shutil.copytree(ov, ov_dst)
    on = YOLO(best).export(format="onnx", imgsz=sz, opset=12, dynamic=False, simplify=True, nms=False, max_det=MAX_DET)
    on_dst = f"{WORK}/export/{NAME}_{sz}.onnx"
    shutil.copy(on, on_dst)
    exports[sz] = {"openvino": ov_dst, "onnx": on_dst, "onnx_sha256": sha256(on_dst)}
    print(sz, "OK")
json.dump(exports, open(f"{WORK}/export/exports.json", "w"), indent=1)


In [ ]:
# Cell 7 — đóng gói
pkg = f"{WORK}/pkg_{NAME}"
if os.path.exists(pkg): shutil.rmtree(pkg)
os.makedirs(pkg)
shutil.copytree(f"{run_dir}/weights", f"{pkg}/weights")
for f in ["results.csv", "args.yaml"]:
    if os.path.exists(f"{run_dir}/{f}"): shutil.copy(f"{run_dir}/{f}", pkg)
for f in glob.glob(f"{run_dir}/*.png") + glob.glob(f"{PROJECT}/{NAME}_val/*.png"):
    shutil.copy(f, pkg)
shutil.copytree(f"{WORK}/export", f"{pkg}/export")
for f in ["val_metrics.json", "dataset_manifest.json"]:
    shutil.copy(f"{WORK}/{f}", pkg)
hashes = {os.path.relpath(p, pkg): sha256(p) for p in glob.glob(f"{pkg}/**/*", recursive=True) if os.path.isfile(p)}
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
log = f"""# TRAIN_LOG — {NAME}
- Ngày (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}
- Model: {MODEL} | imgsz {IMGSZ} | epochs yêu cầu {EPOCHS} | batch {BATCH} | seed {SEED} | deterministic=True | device=0
- GPU: {gpu} | ultralytics {ultralytics.__version__} | torch {torch.__version__} | cuda {torch.version.cuda} | python {platform.python_version()}
- Thời gian train: {train_hours:.2f} h | tổng notebook: {(time.time()-t_start)/3600:.2f} h
- Dữ liệu: VisDrone2019-DET (Ultralytics VisDrone.yaml, GitHub release) — xem dataset_manifest.json
- Val: mAP50 {val_metrics['mAP50']:.4f} | mAP50-95 {val_metrics['mAP50_95']:.4f}
- Export: OpenVINO FP16 + ONNX opset12 tại imgsz {EXPORT_SIZES}; YOLO26 end-to-end (không NMS)
- Lưu ý: CUDA vẫn có nondeterminism nhỏ dù deterministic=True.

## SHA256
""" + "\n".join(f"{v}  {k}" for k, v in sorted(hashes.items()))
open(f"{pkg}/TRAIN_LOG.md", "w").write(log)
zip_path = shutil.make_archive(f"{WORK}/28_{NAME}_export", "zip", pkg)
print(zip_path, f"{os.path.getsize(zip_path)/1e6:.0f} MB", "sha256", sha256(zip_path))
shutil.rmtree(DATASETS_DIR, ignore_errors=True)   # Output nhẹ: giữ runs + zip
print("XONG — tải file zip từ tab Output.")
